# 02 — DSP Pipeline
// Filter design & frequency analysis for Pipeline B

**Objective**: Design, analyze, and apply digital filters for audio preprocessing.

Covers:
- FIR bandpass filter design and analysis
- IIR Butterworth filter design and analysis
- FIR vs IIR comparison
- Pre-emphasis, normalization, silence removal
- Before/after signal comparison

---

## Mathematical Foundation

### Discrete-Time Signal Representation

A sampled audio signal at sampling rate $f_s = 22050$ Hz:

$$x[n] = x_c(nT_s), \quad T_s = \frac{1}{f_s}, \quad n = 0, 1, \dots, N-1$$

where $N = 88200$ samples (4 seconds × 22050 Hz).

### Nyquist Frequency

$$f_{\text{Nyq}} = \frac{f_s}{2} = 11025 \text{ Hz}$$

All filter cutoffs are normalized to $f_{\text{Nyq}}$:

$$\hat{f} = \frac{f}{f_{\text{Nyq}}}$$

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")


import numpy as np
import matplotlib.pyplot as plt

import config
from src.data_loader import load_metadata, load_audio, get_file_path
from src.dsp_pipeline import (
    design_fir_bandpass, apply_fir_filter, fir_frequency_response,
    fir_group_delay, fir_impulse_response,
    design_iir_bandpass, apply_iir_filter, iir_frequency_response,
    iir_pole_zero, check_stability,
    pre_emphasis, normalize_amplitude, remove_silence,
    pipeline_b_process, compare_before_after
)
from src.visualization import plot_filter_response, plot_pole_zero, plot_before_after

plt.style.use('dark_background')
%matplotlib inline

## 1. FIR Bandpass Filter

### Theory: Window Method (Hann Window)

The ideal bandpass impulse response:

$$h_d[n] = \frac{\sin(\omega_{h} n)}{\pi n} - \frac{\sin(\omega_{l} n)}{\pi n}, \quad \omega_l = \frac{2\pi f_l}{f_s}, \quad \omega_h = \frac{2\pi f_h}{f_s}$$

where $f_l = 50$ Hz and $f_h = 10000$ Hz.

Apply Hann window to truncate to $M = 101$ taps:

$$w[n] = 0.5 \left(1 - \cos\left(\frac{2\pi n}{M-1}\right)\right), \quad n = 0, 1, \dots, M-1$$

Final FIR coefficients:

$$h[n] = h_d[n] \cdot w[n]$$

### Transfer Function (Z-domain)

$$H(z) = \sum_{k=0}^{M-1} h[k] \, z^{-k}$$

### Frequency Response

Evaluate $H(z)$ on the unit circle $z = e^{j\omega}$:

$$H(e^{j\omega}) = \sum_{k=0}^{M-1} h[k] \, e^{-j\omega k}$$

- **Magnitude**: $|H(e^{j\omega})|$
- **Phase**: $\angle H(e^{j\omega})$ — **linear** for symmetric FIR → $\phi(\omega) = -\frac{M-1}{2}\omega$

### Zero-Phase Filtering (filtfilt)

Forward-backward filtering eliminates phase distortion:

$$y[n] = \mathcal{F}^{-1}\left\{ |H(e^{j\omega})|^2 \cdot X(e^{j\omega}) \right\}$$

Effective magnitude response is **squared**: $|H_{\text{eff}}|^2 = |H|^2$, phase = 0.

In [ ]:
# Design FIR filter
fir_coeffs = design_fir_bandpass()
print(f"FIR filter order: {len(fir_coeffs) - 1}")
print(f"Number of taps: {len(fir_coeffs)}")

# Frequency & phase response
w, mag_db, phase = fir_frequency_response(fir_coeffs)
fig = plot_filter_response(w, mag_db, phase, title='FIR Bandpass Filter',
                           save_name='fig_fir_frequency_response.png')
plt.show()

# Group delay
w_gd, gd = fir_group_delay(fir_coeffs)
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')
ax.plot(w_gd, gd, color='#a855f7')
ax.set_xlabel('Frequency (Hz)', color='#888888')
ax.set_ylabel('Group Delay (samples)', color='#888888')
ax.set_title('FIR Group Delay', color='#ffffff')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

# Impulse response
ir = fir_impulse_response(fir_coeffs)
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#111111')
ax.stem(ir, linefmt='#00ff41', markerfmt='o', basefmt='#888888')
ax.set_title('FIR Impulse Response', color='#ffffff')
ax.set_xlabel('Sample', color='#888888')
plt.tight_layout()
plt.show()

## 2. IIR Butterworth Filter

### Butterworth Magnitude Response

An $N$-th order Butterworth lowpass has maximally flat passband:

$$|H_a(j\Omega)|^2 = \frac{1}{1 + \left(\frac{\Omega}{\Omega_c}\right)^{2N}}$$

For this project: $N = 5$ (order), bandpass version via lowpass-to-bandpass transformation.

### Analog Butterworth Poles

Poles of the normalized Butterworth polynomial lie on the unit circle in the left half-plane:

$$s_k = e^{j\pi\frac{2k + N - 1}{2N}}, \quad k = 1, 2, \dots, N$$

### Bilinear Transform (Analog → Digital)

Convert analog filter $H_a(s)$ to digital $H(z)$:

$$s = \frac{2}{T} \cdot \frac{1 - z^{-1}}{1 + z^{-1}}$$

with frequency pre-warping to correct nonlinear mapping:

$$\Omega_c = \frac{2}{T} \tan\left(\frac{\omega_c}{2}\right)$$

### IIR Transfer Function

$$H(z) = \frac{\sum_{k=0}^{M} b_k \, z^{-k}}{\sum_{k=0}^{N} a_k \, z^{-k}} = \frac{B(z)}{A(z)}$$

### Stability Criterion

The system is **stable** iff all poles of $H(z)$ lie inside the unit circle:

$$|p_i| < 1, \quad \forall \, i$$

### Why FIR over IIR for this project?

| Property | FIR | IIR |
|----------|-----|-----|
| Phase | $\phi(\omega) = -\frac{M-1}{2}\omega$ (linear) | Non-linear $\phi(\omega)$ |
| Stability | Always stable (no poles) | Must verify $\|p_i\| < 1$ |
| Order needed | $M = 101$ | $N = 5$ |

**Decision**: FIR chosen because linear phase preserves temporal structure of audio events.

In [ ]:
# Design IIR filter
b, a = design_iir_bandpass()
print(f"IIR b coefficients: {len(b)}")
print(f"IIR a coefficients: {len(a)}")
print(f"Stable: {check_stability(a)}")

# Frequency & phase response
w_iir, mag_db_iir, phase_iir = iir_frequency_response(b, a)
fig = plot_filter_response(w_iir, mag_db_iir, phase_iir, title='IIR Butterworth Filter',
                           save_name='fig_iir_frequency_response.png')
plt.show()

# Pole-zero plot
zeros, poles = iir_pole_zero(b, a)
fig = plot_pole_zero(zeros, poles, title='IIR Butterworth Pole-Zero',
                     save_name='fig_iir_pole_zero.png')
plt.show()

## 3. FIR vs IIR Comparison

### Group Delay

Group delay measures the time delay of each frequency component:

$$\tau_g(\omega) = -\frac{d\phi(\omega)}{d\omega}$$

- **FIR** (linear phase): $\tau_g = \frac{M-1}{2} = 50$ samples (constant for all frequencies)
- **IIR**: $\tau_g(\omega)$ varies with frequency → different frequencies arrive at different times

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
fig.patch.set_facecolor('#0a0a0a')

# Magnitude comparison
ax1.plot(w, mag_db, color='#00ff41', label='FIR (order 101)')
ax1.plot(w_iir, mag_db_iir, color='#ff0080', label='IIR Butterworth (order 5)')
ax1.set_ylabel('Magnitude (dB)', color='#888888')
ax1.set_title('FIR vs IIR — Magnitude Response', color='#00ffff')
ax1.legend()
ax1.set_facecolor('#111111')
ax1.grid(True, alpha=0.2)
ax1.axhline(-3, color='#ffaa00', linestyle='--', alpha=0.5)

# Phase comparison
ax2.plot(w, phase, color='#00ff41', label='FIR')
ax2.plot(w_iir, phase_iir, color='#ff0080', label='IIR')
ax2.set_xlabel('Frequency (Hz)', color='#888888')
ax2.set_ylabel('Phase (rad)', color='#888888')
ax2.set_title('FIR vs IIR — Phase Response', color='#00ffff')
ax2.legend()
ax2.set_facecolor('#111111')
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{config.FIGURES_DIR}/fig_fir_vs_iir_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

## 4. Before vs After — Apply to Real Audio

### Pipeline B Processing Chain

$$x[n] \xrightarrow{\text{FIR bandpass}} x_f[n] \xrightarrow{\text{pre-emphasis}} x_p[n] \xrightarrow{\text{normalize}} x_{\text{out}}[n]$$

**Step 1 — FIR Bandpass** (50–10000 Hz):

$$x_f[n] = \text{filtfilt}(h, x[n])$$

**Step 2 — Pre-emphasis** ($\alpha = 0.97$):

$$x_p[n] = x_f[n] - \alpha \cdot x_f[n-1]$$

This is a first-order high-pass filter that boosts high frequencies. Transfer function:

$$H_{\text{pre}}(z) = 1 - \alpha z^{-1}$$

**Step 3 — Peak Normalization**:

$$x_{\text{out}}[n] = \frac{x_p[n]}{\max_n |x_p[n]|}$$

### Signal-to-Noise Ratio (SNR)

$$\text{SNR (dB)} = 10 \log_{10}\left(\frac{P_{\text{signal}}}{P_{\text{noise}}}\right) = 10 \log_{10}\left(\frac{\sum x_{\text{signal}}^2[n]}{\sum x_{\text{noise}}^2[n]}\right)$$

$$\Delta\text{SNR} = \text{SNR}_{\text{processed}} - \text{SNR}_{\text{raw}}$$

A positive $\Delta\text{SNR}$ means Pipeline B improved the signal quality.

In [ ]:
metadata = load_metadata()

# Process one sample from each class
for cls_name in config.CLASS_NAMES:
    row = metadata[metadata['class'] == cls_name].iloc[0]
    y_raw = load_audio(get_file_path(row))
    y_processed = pipeline_b_process(y_raw)
    
    comparison = compare_before_after(y_raw, y_processed)
    print(f"{cls_name}: SNR raw={comparison['snr_raw']:.1f}dB → processed={comparison['snr_processed']:.1f}dB "
          f"(Δ={comparison['snr_improvement']:+.1f}dB)")
    
    fig = plot_before_after(y_raw, y_processed, title=cls_name,
                           save_name=f'fig_before_after_{cls_name}.png')
    plt.show()